<a href="https://colab.research.google.com/github/zk025815-beep/HCCDA-AI-Course/blob/main/AI_Appointment_Booking_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-api-python-client google-auth-httplib2 google-auth-oauthlib

import os
from urllib.parse import urlparse, parse_qs
from google.colab import files
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

SCOPES = [
    "https://www.googleapis.com/auth/calendar",
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/gmail.send",
]

# 1. Upload credentials.json if not already in Colab
if not os.path.exists("credentials.json"):
    print("Click 'Choose Files' below and select your credentials.json file:")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn != "credentials.json":
            os.rename(fn, "credentials.json")

# 2. Authenticate using Colab-compatible manual redirect copy
creds = None
if os.path.exists("token.json"):
    creds = Credentials.from_authorized_user_file("token.json", SCOPES)

if not creds or not creds.valid:
    flow = InstalledAppFlow.from_client_secrets_file(
        "credentials.json", SCOPES, redirect_uri="http://localhost:8080/"
    )
    auth_url, _ = flow.authorization_url(prompt="consent", access_type="offline")

    print("\n1. Click this link to authorize your Google account:\n", auth_url)
    print("\n2. After clicking 'Allow', your browser will show 'This site can’t be reached' (localhost:8080).")
    print("3. Copy the FULL URL from your browser's address bar and paste it below:")

    redirect_response = input("Paste full localhost URL here: ").strip()
    code = parse_qs(urlparse(redirect_response).query)["code"][0]
    flow.fetch_token(code=code)
    creds = flow.credentials

    with open("token.json", "w") as token_file:
        token_file.write(creds.to_json())
    print("\nSuccess! token.json saved.")

# 3. Initialize API Services
calendar_service = build("calendar", "v3", credentials=creds)
sheets_service = build("sheets", "v4", credentials=creds)
gmail_service = build("gmail", "v1", credentials=creds)
print("All 3 Google APIs (Calendar, Sheets, Gmail) are connected and ready!")

Click 'Choose Files' below and select your credentials.json file:


Saving credentials.json to credentials.json

1. Click this link to authorize your Google account:
 https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=327592694821-vnijlhech6lnpbdbcsi2s2epr9hecnd8.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcalendar+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fspreadsheets+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.send&state=qTZ4cqBauvynmBheoZzSyQtYb6af7A&code_challenge=Tp-FhiI6QJlPw0TDJVRIc_LhFJoksoK2YfEenhHOxP8&code_challenge_method=S256&prompt=consent&access_type=offline

2. After clicking 'Allow', your browser will show 'This site can’t be reached' (localhost:8080).
3. Copy the FULL URL from your browser's address bar and paste it below:


In [ ]:
import base64
from datetime import datetime, timedelta
from email.mime.text import MIMEText
import pytz

# ==============================================================================
# CONFIGURATION: Paste your Google Sheet ID below!
# ==============================================================================
SPREADSHEET_ID = "1fPCCECo7NI_oYgOq21LMpKtmfbWTI5lY3K_a238lYWk"  # <-- REPLACE THIS!
TIMEZONE = "Asia/Karachi"  # Set to your local timezone (PKT)
LOCAL_TZ = pytz.timezone(TIMEZONE)

# Business hours: 9:00 AM to 5:00 PM (1-hour appointment slots)
BUSINESS_START_HOUR = 9
BUSINESS_END_HOUR = 17


# ------------------------------------------------------------------------------
# 1. Check Available Slots in Google Calendar (Requirement 4)
# ------------------------------------------------------------------------------
def get_available_slots(date_str: str) -> list:
    """Checks Google Calendar for free 1-hour slots on a specific date (YYYY-MM-DD)."""
    try:
        target_date = datetime.strptime(date_str, "%Y-%m-%d")
    except ValueError:
        return ["Error: Invalid date format. Please use YYYY-MM-DD."]

    # Define start and end of the business day in local timezone
    day_start = LOCAL_TZ.localize(
        target_date.replace(hour=BUSINESS_START_HOUR, minute=0, second=0)
    )
    day_end = LOCAL_TZ.localize(
        target_date.replace(hour=BUSINESS_END_HOUR, minute=0, second=0)
    )

    # Query Google Calendar freebusy API
    body = {
        "timeMin": day_start.isoformat(),
        "timeMax": day_end.isoformat(),
        "timeZone": TIMEZONE,
        "items": [{"id": "primary"}],
    }
    events_result = calendar_service.freebusy().query(body=body).execute()
    busy_intervals = events_result["calendars"]["primary"]["busy"]

    # Generate all possible 1-hour slots between 9 AM and 5 PM
    available_slots = []
    current_slot = day_start

    while current_slot + timedelta(hours=1) <= day_end:
        slot_end = current_slot + timedelta(hours=1)
        is_Conflict = False

        for busy in busy_intervals:
            busy_start = datetime.fromisoformat(
                busy["start"].replace("Z", "+00:00")
            ).astimezone(LOCAL_TZ)
            busy_end = datetime.fromisoformat(
                busy["end"].replace("Z", "+00:00")
            ).astimezone(LOCAL_TZ)

            # Check overlap
            if max(current_slot, busy_start) < min(slot_end, busy_end):
                is_Conflict = True
                break

        if not is_Conflict:
            available_slots.append(current_slot.strftime("%H:%M"))

        current_slot = slot_end

    return available_slots


# ------------------------------------------------------------------------------
# 2. Create Event in Google Calendar (Requirement 6)
# ------------------------------------------------------------------------------
def create_calendar_event(
    customer_name: str, email: str, service: str, date_str: str, time_str: str
) -> str:
    """Creates a real 1-hour appointment on Google Calendar."""
    start_dt = LOCAL_TZ.localize(
        datetime.strptime(f"{date_str} {time_str}", "%Y-%m-%d %H:%M")
    )
    end_dt = start_dt + timedelta(hours=1)

    event_body = {
        "summary": f"{service} - {customer_name}",
        "description": (
            f"Booked via AI Assistant.\nCustomer: {customer_name}\nEmail:"
            f" {email}\nService: {service}"
        ),
        "start": {"dateTime": start_dt.isoformat(), "timeZone": TIMEZONE},
        "end": {"dateTime": end_dt.isoformat(), "timeZone": TIMEZONE},
        "attendees": [{"email": email}],
    }

    event = (
        calendar_service.events()
        .insert(calendarId="primary", body=event_body)
        .execute()
    )
    return event.get("htmlLink")


# ------------------------------------------------------------------------------
# 3. Save Booking Row to Google Sheets (Requirement 7)
# ------------------------------------------------------------------------------
def save_to_google_sheets(
    customer_name: str, email: str, service: str, date_str: str, time_str: str
):
    """Appends a new booking record into the Google Sheet."""
    timestamp = datetime.now(LOCAL_TZ).strftime("%Y-%m-%d %H:%M:%S")
    row_values = [[
        timestamp,
        customer_name,
        email,
        service,
        date_str,
        time_str,
        "Confirmed",
    ]]

    sheets_service.spreadsheets().values().append(
        spreadsheetId=SPREADSHEET_ID,
        range="A:G",
        valueInputOption="USER_ENTERED",
        body={"values": row_values},
    ).execute()


# ------------------------------------------------------------------------------
# 4. Send Confirmation Email via Gmail OAuth (Requirement 8)
# ------------------------------------------------------------------------------
def send_confirmation_email(
    customer_name: str,
    email: str,
    service: str,
    date_str: str,
    time_str: str,
    cal_link: str,
):
    """Sends a real confirmation email to the customer using the Gmail API."""
    subject = f"Appointment Confirmed: {service} on {date_str}"
    body_text = (
        f"Hello {customer_name},\n\n"
        f"Your appointment has been successfully booked!\n\n"
        f"Details:\n"
        f"- Service: {service}\n"
        f"- Date: {date_str}\n"
        f"- Time: {time_str} ({TIMEZONE})\n"
        f"- Calendar Link: {cal_link}\n\n"
        f"Thank you for choosing our clinic!\n"
        f"Best regards,\nAI Booking Assistant"
    )

    message = MIMEText(body_text)
    message["to"] = email
    message["subject"] = subject

    raw_message = base64.urlsafe_b64encode(message.as_bytes()).decode("utf-8")
    gmail_service.users().messages().send(
        userId="me", body={"raw": raw_message}
    ).execute()


# ------------------------------------------------------------------------------
# 5. Master Booking Function (Chains Calendar + Sheets + Gmail)
# ------------------------------------------------------------------------------
def book_appointment_pipeline(
    customer_name: str, email: str, service: str, date_str: str, time_str: str
) -> dict:
    """Checks if slot is still free, then creates Calendar event, logs to Sheet, and sends Gmail."""
    free_slots = get_available_slots(date_str)
    if time_str not in free_slots:
        return {
            "status": "error",
            "message": (
                f"Slot {time_str} on {date_str} is not available. Free slots:"
                f" {free_slots}"
            ),
        }

    cal_link = create_calendar_event(
        customer_name, email, service, date_str, time_str
    )
    save_to_google_sheets(customer_name, email, service, date_str, time_str)
    send_confirmation_email(
        customer_name, email, service, date_str, time_str, cal_link
    )

    return {
        "status": "success",
        "message": (
            f"Appointment confirmed for {customer_name} on {date_str} at"
            f" {time_str}."
        ),
        "calendar_link": cal_link,
    }


print("Cell 2 loaded: All Google Workspace tool functions are ready!")

In [ ]:
# Check available slots for tomorrow
tomorrow = (datetime.now(LOCAL_TZ) + timedelta(days=1)).strftime("%Y-%m-%d")
slots = get_available_slots(tomorrow)
print(f"Available 1-hour slots on {tomorrow}:")
print(slots)

In [ ]:
# ==============================================================================
# Cell 4: Install Dependencies & Setup RAG Pipeline (ChromaDB + Embeddings)
# ==============================================================================
!pip install -q openai chromadb sentence-transformers gradio

import chromadb
from chromadb.utils import embedding_functions

# 1. Initialize ChromaDB in-memory client with free local HuggingFace embeddings
chroma_client = chromadb.Client()
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Reset collection if re-running cell
try:
    chroma_client.delete_collection("clinic_knowledge_base")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="clinic_knowledge_base",
    embedding_function=embedding_fn
)

# 2. Clinic Business Information, Timings, Services, and Policies
clinic_docs = [
    "Clinic Name & Location: CarePlus Medical & Dental Clinic, located at Main Boulevard, Gulgasht Colony, Multan. Contact phone: +92-300-1234567.",
    "Operating Hours & Timings: We are open Monday to Saturday from 9:00 AM to 5:00 PM PKT (Asia/Karachi). All appointments are 1-hour slots starting on the hour (09:00, 10:00, 11:00, 12:00, 13:00, 14:00, 15:00, 16:00). We are closed on Sundays.",
    "Services & Pricing: 1) General Consultation (Rs. 2,000) - Complete physical checkup and diagnosis. 2) Dental Checkup & Scaling (Rs. 3,500) - Teeth cleaning, plaque removal, and dental exam. 3) Skin & Dermatology Consultation (Rs. 2,500) - Acne, allergy, and skin treatment. 4) Physiotherapy Session (Rs. 3,000) - Joint pain and rehab therapy.",
    "Booking & Cancellation Policy: Customers must provide their full name, email address, service type, and preferred date and time to book an appointment. Cancellations or rescheduling must be done at least 24 hours in advance. No-shows may be charged a 50% booking fee.",
    "FAQ - Insurance & Payment: We accept cash, bank transfer, SadaPay, JazzCash, and major credit/debit cards. We provide stamped medical invoices for corporate insurance reimbursement.",
    "FAQ - Preparation & Arrival: Please arrive 10 minutes prior to your scheduled slot and bring any previous medical reports or prescriptions."
]

collection.add(
    documents=clinic_docs,
    ids=[f"doc_{i}" for i in range(len(clinic_docs))]
)

def search_clinic_knowledge_base(query: str) -> str:
    """Queries the RAG vector store for clinic policies, timings, services, and FAQs."""
    results = collection.query(query_texts=[query], n_results=3)
    retrieved_chunks = results["documents"][0]
    return "\n\n".join(retrieved_chunks)

print("Cell 4 loaded: RAG Knowledge Base is indexed and ready!")
print("Sample RAG test:", search_clinic_knowledge_base("What services do you offer and what are the prices?"))

In [ ]:
# ==============================================================================
# Cell 5: OpenRouter Function Calling Agent & Gradio Chat Interface
# ==============================================================================
import json
from datetime import datetime
from openai import OpenAI
import gradio as gr
from google.colab import userdata
userdata.get('secretName')

# ------------------------------------------------------------------------------
# CONFIGURATION: Paste your OpenRouter API Key here!
# ------------------------------------------------------------------------------
OPENROUTER_API_KEY = "OPENROUTER_API_KEY"  # <-- REPLACE THIS (sk-or-v1-...)
OPENROUTER_MODEL = "openai/gpt-4o-mini"         # Or "google/gemini-2.0-flash-001" / "openrouter/free"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# ------------------------------------------------------------------------------
# Tool Schema Definitions for OpenRouter (OpenAI-Compatible Format)
# ------------------------------------------------------------------------------
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_clinic_knowledge_base",
            "description": "Search the clinic RAG knowledge base for business info, timings, services, pricing, location, and policies/FAQs.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The question or search topic (e.g., 'services and prices', 'cancellation policy', 'clinic timings')."
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_available_slots",
            "description": "Check Google Calendar for available 1-hour appointment time slots on a specific date.",
            "parameters": {
                "type": "object",
                "properties": {
                    "date_str": {
                        "type": "string",
                        "description": "The date to check in YYYY-MM-DD format (e.g., '2026-11-05')."
                    }
                },
                "required": ["date_str"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "book_appointment_pipeline",
            "description": "Book and confirm an appointment ONLY AFTER collecting customer_name, email, service, date_str, and time_str, and AFTER the customer confirms an available slot.",
            "parameters": {
                "type": "object",
                "properties": {
                    "customer_name": {
                        "type": "string",
                        "description": "Full name of the customer."
                    },
                    "email": {
                        "type": "string",
                        "description": "Email address of the customer for calendar invite and Gmail confirmation."
                    },
                    "service": {
                        "type": "string",
                        "description": "Name of the clinic service being booked (e.g., 'Dental Checkup & Scaling', 'General Consultation')."
                    },
                    "date_str": {
                        "type": "string",
                        "description": "Appointment date in YYYY-MM-DD format."
                    },
                    "time_str": {
                        "type": "string",
                        "description": "24-hour start time slot in HH:MM format (e.g., '10:00', '14:00')."
                    }
                },
                "required": ["customer_name", "email", "service", "date_str", "time_str"]
            }
        }
    }
]

# ------------------------------------------------------------------------------
# Conversation & Function Calling Orchestrator
# ------------------------------------------------------------------------------
def chat_with_assistant(user_message, history):
    today_str = datetime.now(LOCAL_TZ).strftime("%A, %Y-%m-%d")

    system_prompt = (
        f"You are the AI Appointment Booking Assistant for CarePlus Medical & Dental Clinic.\n"
        f"Current Date: {today_str} (Timezone: {TIMEZONE}).\n\n"
        f"Follow these rules strictly:\n"
        f"1. For any question about clinic services, prices, timings, location, or policies, ALWAYS call `search_clinic_knowledge_base` first.\n"
        f"2. When a user wants to book an appointment, you MUST collect 4 pieces of information: "
        f"(a) Customer's Full Name, (b) Email Address, (c) Service Type, and (d) Preferred Date/Time.\n"
        f"3. Once the user provides a preferred date, call `get_available_slots(date_str)` to check real-time availability on Google Calendar.\n"
        f"4. Present the available time slots clearly and ask the user to select/confirm their slot (and provide any missing info like name or email).\n"
        f"5. ONLY call `book_appointment_pipeline` once you have all 5 fields (customer_name, email, service, date_str in YYYY-MM-DD, time_str in HH:MM 24-hour format) and the user has agreed to the slot.\n"
        f"6. After booking, share the confirmation details and the Google Calendar event link with the customer."
    )

    messages = [{"role": "system", "content": system_prompt}]

    # Load prior conversation turns from Gradio history
    for turn in history:
        if isinstance(turn, dict):
            messages.append({"role": turn["role"], "content": str(turn["content"])})
        elif isinstance(turn, (list, tuple)) and len(turn) == 2:
            if turn[0]:
                messages.append({"role": "user", "content": str(turn[0])})
            if turn[1]:
                messages.append({"role": "assistant", "content": str(turn[1])})

    messages.append({"role": "user", "content": user_message})

    # Loop up to 4 times so the LLM can chain multiple tool calls if needed
    for _ in range(4):
        response = client.chat.completions.create(
            model=OPENROUTER_MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )

        assistant_msg = response.choices[0].message

        # If no tool call was requested, return the final text response
        if not assistant_msg.tool_calls:
            return assistant_msg.content or "How else can I assist you today?"

        # Append assistant's tool call request to messages
        messages.append(assistant_msg)

        # Execute each requested tool call
        for tool_call in assistant_msg.tool_calls:
            fn_name = tool_call.function.name
            try:
                fn_args = json.loads(tool_call.function.arguments)
            except Exception:
                fn_args = {}

            print(f"[Tool Call] Executing: {fn_name}({fn_args})")

            try:
                if fn_name == "search_clinic_knowledge_base":
                    tool_output = search_clinic_knowledge_base(fn_args.get("query", ""))
                elif fn_name == "get_available_slots":
                    slots = get_available_slots(fn_args.get("date_str", ""))
                    tool_output = json.dumps({"available_slots": slots})
                elif fn_name == "book_appointment_pipeline":
                    result = book_appointment_pipeline(
                        customer_name=fn_args.get("customer_name"),
                        email=fn_args.get("email"),
                        service=fn_args.get("service"),
                        date_str=fn_args.get("date_str"),
                        time_str=fn_args.get("time_str"),
                    )
                    tool_output = json.dumps(result)
                else:
                    tool_output = json.dumps({"error": f"Unknown tool: {fn_name}"})
            except Exception as e:
                tool_output = json.dumps({"error": str(e)})

            # Send tool output back to the LLM
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": fn_name,
                "content": str(tool_output),
            })

    return "I completed the actions, please let me know if you need anything else!"

# ------------------------------------------------------------------------------
# Launch Gradio Interface
# ------------------------------------------------------------------------------
demo = gr.ChatInterface(
    fn=chat_with_assistant,
    title="🏥 CarePlus Clinic — AI Appointment Booking Assistant",
    description=(
        "Ask about our **services, prices, clinic timings, or policies** (powered by RAG), "
        "or **book a real appointment** synced with Google Calendar, Google Sheets, and Gmail!"
    ),
    examples=[
        "What services do you offer and what are your charges?",
        "What are your clinic timings and cancellation policy?",
        "I want to book a Dental Checkup for tomorrow.",
    ],
)

demo.launch(share=True, debug=True)